In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import re
import json
import os
from collections import defaultdict
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# One-time download of NLTK English stop words (telechargement de bibliotheque de NLTK english stop word)
nltk.download('stopwords', quiet=True)

# Original corpus preserved intact (D1–D10) (le corpus qui contient des documents de D1 a D10)
corpus = {
    "D1": "Information retrieval helps users find relevant documents.",
    "D2": "Information Retrieval systems index documents and documents.",
    "D3": "Search engines retrieve relevant web documents.",
    "D4": "Document indexing supports efficient information retrieval.",
    "D5": "Machine learning improves search engine ranking.",
    "D6": "Search systems use indexed documents to retrieve information.",
    "D7": "The search engine retrieves documents; the documents are searchable.",
    "D8": "Indexing documents makes information retrieval more efficient.",
    "D9": "DOCUMENTS contain information about document indexing.",
    "D10": "Retrieval systems organise documents and document collections."
}

print(f"Loaded {len(corpus)} documents.") # Validation de Success de code above

Loaded 10 documents.


**Part 1:A Inspection de Corpus:** Uppercase/lowercase forms: Information/information (D1, D2 vs. D4, D6, D8, D9), Retrieval/retrieval (D2, D10 vs. D1, D4, D8), DOCUMENTS/Document/documents (D9, D4 vs. D1–D3), Search/search (D3, D6 vs. D5, D7), Indexing/indexing (D8 vs. D4, D9), The/the (D7).   Punctuation: Trailing periods (.) on every document, plus a semicolon (;) attached to documents; in D7.   Repeated terms: documents appears twice in D2 (documents and documents) and twice in D7; the also appears twice in D7.   Morphological variants: document/documents, index/indexed/indexing, retrieve/retrieves/retrieval, search/searchable, engine/engines.   

In [4]:
# Part 1.B: Lowercase, remove punctuation, and tokenize (miniscule, elimination de ponctuation et tokenization)
def preprocess_document(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

# Version A: Lowercase + punctuation removal
tokens_A = {doc_id: preprocess_document(text) for doc_id, text in corpus.items()}

# Part 1.C: Stop-word removal (Version B)
stop_words = set(stopwords.words('english'))
tokens_B = {
    doc_id: [w for w in tokens if w not in stop_words]
    for doc_id, tokens in tokens_A.items()
}

# Part 1.D: Porter stemming (Version C)
stemmer = PorterStemmer()
tokens_C = {
    doc_id: [stemmer.stem(w) for w in tokens]
    for doc_id, tokens in tokens_B.items()
}

# Inspect outputs across all three versions
for label, version in [("Version A (Clean)", tokens_A),
                       ("Version B (+ No Stopwords)", tokens_B),
                       ("Version C (+ Porter Stemming)", tokens_C)]:
    print(f"\n=== {label} ===")
    for doc_id, toks in version.items():
        print(f"{doc_id}: {toks}")


=== Version A (Clean) ===
D1: ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
D2: ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents']
D3: ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
D4: ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
D5: ['machine', 'learning', 'improves', 'search', 'engine', 'ranking']
D6: ['search', 'systems', 'use', 'indexed', 'documents', 'to', 'retrieve', 'information']
D7: ['the', 'search', 'engine', 'retrieves', 'documents', 'the', 'documents', 'are', 'searchable']
D8: ['indexing', 'documents', 'makes', 'information', 'retrieval', 'more', 'efficient']
D9: ['documents', 'contain', 'information', 'about', 'document', 'indexing']
D10: ['retrieval', 'systems', 'organise', 'documents', 'and', 'document', 'collections']

=== Version B (+ No Stopwords) ===
D1: ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
D2: ['information

In [5]:
# construction de vocabulaire ordonnée
def build_vocabulary(tokenized_documents):
    vocab = set()
    for tokens in tokenized_documents.values():
        vocab.update(tokens)
    return sorted(vocab)

# construction de matrice d'incidence
def build_term_document_matrix(tokenized_documents):
    vocab = build_vocabulary(tokenized_documents)
    doc_ids = list(tokenized_documents.keys())
    matrix = {}
    for term in vocab:
        matrix[term] = [1 if term in tokenized_documents[doc_id] else 0 for doc_id in doc_ids]
    return vocab, doc_ids, matrix

# construction de matrice des termes
def build_tf_matrix(tokenized_documents):
    vocab = build_vocabulary(tokenized_documents)
    doc_ids = list(tokenized_documents.keys())
    matrix = {}
    for term in vocab:
        matrix[term] = [tokenized_documents[doc_id].count(term) for doc_id in doc_ids]
    return vocab, doc_ids, matrix

# fonction de representation sur forme d'un table avec export de notre matrix a un fichier CSV
def matrix_to_dataframe(matrix, doc_ids):
    # pandas used strictly for display and CSV export
    return pd.DataFrame.from_dict(matrix, orient='index', columns=doc_ids)

versions = {"A": tokens_A, "B": tokens_B, "C": tokens_C}
vocabularies = {}
binary_matrices = {}
tf_matrices = {}

for ver, toks in versions.items():
    vocab, doc_ids, bin_mat = build_term_document_matrix(toks)
    _, _, tf_mat = build_tf_matrix(toks)
    vocabularies[ver] = vocab
    binary_matrices[ver] = (doc_ids, bin_mat)
    tf_matrices[ver] = (doc_ids, tf_mat)

# affichage de taille de chaque vocabulaire de chaque version
print(f"Vocabulary sizes -> A: {len(vocabularies['A'])}, B: {len(vocabularies['B'])}, C: {len(vocabularies['C'])}")

# execution de fonction de representation dans le cas de matrice binaire des termes
print("\n--- Binary Term-Document Matrix (Version C) ---")
display(matrix_to_dataframe(binary_matrices["C"][1], binary_matrices["C"][0]))

# execution de fonction de representation dans le cas de matrice term frequency regulaire
print("\n--- Term-Frequency Matrix (Version C) ---")
display(matrix_to_dataframe(tf_matrices["C"][1], tf_matrices["C"][0]))

Vocabulary sizes -> A: 36, B: 30, C: 24

--- Binary Term-Document Matrix (Version C) ---


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
collect,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,1,1,1,1,0,1,1,1,1,1
effici,0,0,0,1,0,0,0,1,0,0
engin,0,0,1,0,1,0,1,0,0,0
find,1,0,0,0,0,0,0,0,0,0
help,1,0,0,0,0,0,0,0,0,0
improv,0,0,0,0,1,0,0,0,0,0
index,0,1,0,1,0,1,0,1,1,0
inform,1,1,0,1,0,1,0,1,1,0



--- Term-Frequency Matrix (Version C) ---


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
collect,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,1,2,1,1,0,1,2,1,2,2
effici,0,0,0,1,0,0,0,1,0,0
engin,0,0,1,0,1,0,1,0,0,0
find,1,0,0,0,0,0,0,0,0,0
help,1,0,0,0,0,0,0,0,0,0
improv,0,0,0,0,1,0,0,0,0,0
index,0,1,0,1,0,1,0,1,1,0
inform,1,1,0,1,0,1,0,1,1,0


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


**Comparison:**
*   **Binary matrix:** Records only term presence (1) or absence (0), which supports Boolean retrieval (AND, OR, NOT) but treats a single mention and repeated mentions identically.

*   **Term-frequency (TF) matrix:** Records the exact count of occurrences per document (e.g., documents has value 2 in D2 and D7, and document after stemming has value 2 in D2, D7, D9, and D10), capturing local term importance needed for ranked retrieval models like TF-IDF.

**Index Manuelle (Version A):**

*  **D1 (Version A):** ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']

*  **D2 (Version A):** ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents']

*  **D3 (Version A):** ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']



1.   **information** ---> {'D1': [0], 'D2': [0]}
2.   **retrieval**   ---> {'D1': [1], 'D2': [1]}
3.   **documents**   ---> {'D1': [6], 'D2': [4, 6], 'D3': [5]}



**Index Manuelle (Version B):**

*  **D1 (Version B):** ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']

*  **D2 (Version B):** ['information', 'retrieval', 'systems', 'index', 'documents', 'documents']

*  **D3 (Version B):** ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']

1.   **information** ---> {'D1': [0], 'D2': [0]} | ***identique avec version A***
2.   **retrieval**   ---> {'D1': [1], 'D2': [1]} | ***identique avec version A***
3.   **documents**   ---> {'D1': [6], 'D2': [4, 5], 'D3': [5]}  | ***D2 décale car on a eliminer les stop words***


In [6]:
def build_inverted_index(tokenized_documents):
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in tokenized_documents.items():
        for pos, term in enumerate(tokens):
            index[term][doc_id].append(pos)
    return {term: dict(postings) for term, postings in sorted(index.items())}

def search_term(inverted_index, term):
    return list(inverted_index.get(term, {}).keys())

inverted_indexes = {
    ver: build_inverted_index(toks)
    for ver, toks in versions.items()
}

# Compare auto-generated index on D1–D3 (Version A) with manual index
manual_index_D1_D3_A = {
    "information": {"D1": [0], "D2": [0]},
    "retrieval": {"D1": [1], "D2": [1]},
    "documents": {"D1": [6], "D2": [4, 6], "D3": [5]}
}

auto_D1_D3_A = build_inverted_index({k: tokens_A[k] for k in ["D1", "D2", "D3"]})

for term, expected in manual_index_D1_D3_A.items():
    actual = auto_D1_D3_A.get(term, {})
    print(f"Term '{term}': Manual={expected} | Auto={actual} | Match={actual == expected}")

Term 'information': Manual={'D1': [0], 'D2': [0]} | Auto={'D1': [0], 'D2': [0]} | Match=True
Term 'retrieval': Manual={'D1': [1], 'D2': [1]} | Auto={'D1': [1], 'D2': [1]} | Match=True
Term 'documents': Manual={'D1': [6], 'D2': [4, 6], 'D3': [5]} | Auto={'D1': [6], 'D2': [4, 6], 'D3': [5]} | Match=True


In [7]:
# creation d'un table des statistiques
comparison_rows = []

for ver in ["A", "B", "C"]:
    toks = versions[ver]
    inv_idx = inverted_indexes[ver]

    vocab_size = len(vocabularies[ver])
    avg_tokens = sum(len(t) for t in toks.values()) / len(toks)
    total_postings = sum(len(postings) for postings in inv_idx.values())
    total_positions = sum(
        len(positions)
        for postings in inv_idx.values()
        for positions in postings.values()
    )

    comparison_rows.append({
        "Version": ver,
        "Vocabulary Size": vocab_size,
        "Avg Tokens / Doc": round(avg_tokens, 2),
        "Total Postings": total_postings,
        "Total Stored Positions": total_positions
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,Version,Vocabulary Size,Avg Tokens / Doc,Total Postings,Total Stored Positions
0,A,36,6.9,66,69
1,B,30,6.1,59,61
2,C,24,6.1,57,61


In [8]:
# code de verification de notre travail

# 1 & 2. Verify search_term() matches ground truth across all required test cases
def verify_search(version_name, tokens_dict, inv_idx, query_term):
    result = search_term(inv_idx, query_term)
    expected = [doc_id for doc_id, toks in tokens_dict.items() if query_term in toks]
    assert result == expected, f"[{version_name}] Mismatch for '{query_term}': {result} != {expected}"
    print(f"[PASS] Version {version_name} | search_term('{query_term}') -> {result}")

print("=== Search Verification ===")
verify_search("B", tokens_B, inverted_indexes["B"], "web")          # Occurs in 1 document (D3)
verify_search("B", tokens_B, inverted_indexes["B"], "information")  # Occurs in multiple documents
verify_search("B", tokens_B, inverted_indexes["B"], "documents")    # Form before stemming
verify_search("C", tokens_C, inverted_indexes["C"], "documents")    # No longer matches after stemming
verify_search("C", tokens_C, inverted_indexes["C"], "document")     # Stemmed form merges document + documents
verify_search("C", tokens_C, inverted_indexes["C"], "blockchain")   # Unknown term -> []

# 3. Verify every stored position matches the exact zero-based index in processed tokens
print("\n=== Positional Index Verification ===")
for ver in ["A", "B", "C"]:
    for term, postings in inverted_indexes[ver].items():
        for doc_id, positions in postings.items():
            for pos in positions:
                assert versions[ver][doc_id][pos] == term, (
                    f"Position error in Version {ver}, {doc_id}[{pos}]"
                )
    print(f"[PASS] Version {ver}: All positional indices verified.")

# Demonstrate position shift between A and B in D2
print(f"Position check D2 'documents': Version A={inverted_indexes['A']['documents']['D2']} "
      f"vs Version B={inverted_indexes['B']['documents']['D2']} (shifted after removing 'and')")

# 4. Assert manual vs automatic index on D1–D3
for term, expected in manual_index_D1_D3_A.items():
    assert auto_D1_D3_A[term] == expected
print("[PASS] Manual vs. automatic D1–D3 index matches 100%.")

# 5. Verify Binary and TF matrix values
print("\n=== Matrix Verification ===")
for ver in ["A", "B", "C"]:
    doc_ids, bin_mat = binary_matrices[ver]
    _, tf_mat = tf_matrices[ver]
    for term, row in bin_mat.items():
        assert all(val in (0, 1) for val in row), f"Non-binary value in Version {ver}, term '{term}'"
        for idx, doc_id in enumerate(doc_ids):
            actual_count = versions[ver][doc_id].count(term)
            assert tf_mat[term][idx] == actual_count, f"TF mismatch in Version {ver}, {term}/{doc_id}"
            assert bin_mat[term][idx] == (1 if actual_count > 0 else 0)
    print(f"[PASS] Version {ver}: Binary and TF matrices verified.")

=== Search Verification ===
[PASS] Version B | search_term('web') -> ['D3']
[PASS] Version B | search_term('information') -> ['D1', 'D2', 'D4', 'D6', 'D8', 'D9']
[PASS] Version B | search_term('documents') -> ['D1', 'D2', 'D3', 'D6', 'D7', 'D8', 'D9', 'D10']
[PASS] Version C | search_term('documents') -> []
[PASS] Version C | search_term('document') -> ['D1', 'D2', 'D3', 'D4', 'D6', 'D7', 'D8', 'D9', 'D10']
[PASS] Version C | search_term('blockchain') -> []

=== Positional Index Verification ===
[PASS] Version A: All positional indices verified.
[PASS] Version B: All positional indices verified.
[PASS] Version C: All positional indices verified.
Position check D2 'documents': Version A=[4, 6] vs Version B=[4, 5] (shifted after removing 'and')
[PASS] Manual vs. automatic D1–D3 index matches 100%.

=== Matrix Verification ===
[PASS] Version A: Binary and TF matrices verified.
[PASS] Version B: Binary and TF matrices verified.
[PASS] Version C: Binary and TF matrices verified.


In [12]:
import os
import json
import zipfile
import nltk
import pandas as pd

# 1. Create output directory
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)

# 2. Export tokens, vocabularies, binary/TF matrices, and inverted indexes for A, B, and C
for ver in ["A", "B", "C"]:
    with open(f"{results_dir}/tokens_{ver}.json", "w", encoding="utf-8") as f:
        json.dump(versions[ver], f, indent=2)

    with open(f"{results_dir}/vocabulary_{ver}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(vocabularies[ver]) + "\n")

    doc_ids, bin_mat = binary_matrices[ver]
    matrix_to_dataframe(bin_mat, doc_ids).to_csv(
        f"{results_dir}/binary_matrix_{ver}.csv", index_label="term"
    )

    _, tf_mat = tf_matrices[ver]
    matrix_to_dataframe(tf_mat, doc_ids).to_csv(
        f"{results_dir}/tf_matrix_{ver}.csv", index_label="term"
    )

    with open(f"{results_dir}/inverted_index_{ver}.json", "w", encoding="utf-8") as f:
        json.dump(inverted_indexes[ver], f, indent=2)

# 3. Export preprocessing comparison table
comparison_df.to_csv(f"{results_dir}/preprocessing_comparison.csv", index=False)

# 4. Generate requirements.txt with exact package versions used in Colab
requirements_content = f"nltk=={nltk.__version__}\npandas=={pd.__version__}\n"
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

# 5. Build Markdown comparison table without requiring external 'tabulate'
headers = list(comparison_df.columns)
md_lines = [
    "| " + " | ".join(headers) + " |",
    "| " + " | ".join(["---"] * len(headers)) + " |"
]
for _, row in comparison_df.iterrows():
    md_lines.append("| " + " | ".join(str(row[col]) for col in headers) + " |")
md_table = "\n".join(md_lines)

# 6. Generate README.md (interpretation strictly within 100–150 words)
interpretation = (
    "Each preprocessing stage changes the vocabulary and inverted index in specific ways. "
    "Lowercasing and removing punctuation unify surface variants such as `DOCUMENTS` and `documents;`, "
    "preventing duplicate vocabulary entries without discarding meaningful words. Removing stop words "
    "drops high-frequency function words (`the`, `and`, `to`, `are`, `about`, `more`), reducing average "
    "document length, total postings, and stored token positions while keeping content-bearing terms. "
    "Porter stemming delivers the steepest vocabulary contraction by collapsing morphological variants "
    "such as `document`/`documents`, `index`/`indexed`/`indexing`, and `retrieve`/`retrieves`/`retrieval` "
    "into single stems, which improves recall and further shrinks posting lists. However, preprocessing "
    "involves clear trade-offs: stemming can conflate distinct terms and lower precision, while deleting "
    "stop words shifts token positions and can disrupt exact phrase matching."
)

word_count = len(interpretation.split())
assert 100 <= word_count <= 150, f"Interpretation word count ({word_count}) is outside 100–150 words."

readme_content = f"""# IR TP1: Document Preprocessing, Representation, and Indexing

## Repository Structure
{'`' * 3}text
IR-TP1-Document-Preprocessing-and-Indexing/
├── README.md
├── requirements.txt
├── tp1_preprocessing_indexing.ipynb
└── results/
    ├── tokens_A.json
    ├── tokens_B.json
    ├── tokens_C.json
    ├── vocabulary_A.txt
    ├── vocabulary_B.txt
    ├── vocabulary_C.txt
    ├── binary_matrix_A.csv
    ├── binary_matrix_B.csv
    ├── binary_matrix_C.csv
    ├── tf_matrix_A.csv
    ├── tf_matrix_B.csv
    ├── tf_matrix_C.csv
    ├── inverted_index_A.json
    ├── inverted_index_B.json
    ├── inverted_index_C.json
    └── preprocessing_comparison.csv
{'`' * 3}

## How to Run
1. Install the required external packages:
   {'`' * 3}bash
   pip install -r requirements.txt
   {'`' * 3}
2. Open `tp1_preprocessing_indexing.ipynb` in Google Colab or Jupyter Notebook and run all cells sequentially.
3. The notebook downloads the NLTK English stop-word corpus automatically, runs all verification checks, and exports all generated artifacts into `results/`.

## Implementation Choices
- **Preprocessing (`preprocess_document`):** Implemented using `str.lower()`, regex punctuation stripping (`re.sub(r'[^\\w\\s]', '', text)`), and whitespace tokenization (`split()`). NLTK is used strictly for the English stop-word list (`nltk.corpus.stopwords`) and `PorterStemmer`, while filtering is implemented manually via list comprehensions.
- **Vocabulary & Matrices:** Sorted vocabularies, binary term-document matrices, and term-frequency matrices are computed from scratch in Python. `pandas` is used only for formatting tables and writing CSV exports.
- **Positional Inverted Index & Search:** Built from scratch using `collections.defaultdict` mapping `term -> document_id -> [zero-based positions]`, paired with `search_term()` for document lookup and automated assertions against the manual D1–D3 index.

## Main Results
{md_table}

## Interpretation: Effect of Preprocessing on Vocabulary and Index
{interpretation}
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

# 7. Verify all expected deliverables exist before zipping
expected_files = [
    "README.md",
    "requirements.txt",
    "results/preprocessing_comparison.csv",
] + [
    f"results/{prefix}_{ver}.{ext}"
    for ver in ["A", "B", "C"]
    for prefix, ext in [
        ("tokens", "json"),
        ("vocabulary", "txt"),
        ("binary_matrix", "csv"),
        ("tf_matrix", "csv"),
        ("inverted_index", "json"),
    ]
]

missing = [path for path in expected_files if not os.path.exists(path)]
assert not missing, f"Missing expected files: {missing}"

# 8. Bundle README.md, requirements.txt, and results/ into one archive
zip_filename = "IR-TP1-deliverables.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in expected_files:
        zf.write(path)

print(f"[PASS] All {len(expected_files)} deliverable files verified.")
print(f"[PASS] README interpretation word count: {word_count} words (target: 100–150).")
print(f"[DONE] Created {zip_filename} containing README.md, requirements.txt, and results/.")

# Trigger automatic download in Google Colab
try:
    from google.colab import files
    files.download(zip_filename)
except ImportError:
    print("Not running in Google Colab; locate the zip file in your working directory.")

[PASS] All 18 deliverable files verified.
[PASS] README interpretation word count: 113 words (target: 100–150).
[DONE] Created IR-TP1-deliverables.zip containing README.md, requirements.txt, and results/.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>